In [2]:
import torch 
from torch import nn
from torch import optim
from protgen.models.transformer import Transformer
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

Load data

In [3]:
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get train tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")

#Get validation tensors
X_val = val_data.tokens
y_val_labels = val_data.labels 
y_val_shifted = val_data.tokens_shifted

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])


Data Loaders

In [4]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

val_loader = DataLoader(val_data, batch_size=32, shuffle=False)


Loss function

In [6]:
transformer= Transformer()

cross_entropy_loss = nn.CrossEntropyLoss(ignore_index=-1) #ignore padding tokens

optimizer = optim.Adam(transformer.parameters(), lr=0.001)


Training loop

In [ ]:
for epoch in range(20):
    transformer.train()

    train_loss_accumulated, num_batches = 0.0, 0
    for x, x_shifted, _ , _ in train_loader:
        optimizer.zero_grad()
        logits= transformer(x) # (batch of proteins, max_seq_len, vocab_size)
        logits_reshaped= logits.reshape(-1, logits.size(-1)) # (batch of proteins * max_seq_len, vocab_size)

        targets= x_shifted.reshape(-1)
        loss= cross_entropy_loss(logits_reshaped, targets)

        loss.backward()

        optimizer.step()

        train_loss_accumulated += loss.item()
        num_batches += 1
        

    train_loss_avg = train_loss_accumulated / num_batches
        

    transformer.eval()

    with torch.no_grad():
        val_loss_accumulated, num_batches = 0.0, 0
        for x, x_shifted, _ , _ in val_loader:
            logits= transformer(x) # (batch of proteins, max_seq_len, vocab_size)
            logits_reshaped= logits.reshape(-1, logits.size(-1)) # (batch of proteins * max_seq_len, vocab_size)

            targets= x_shifted.reshape(-1)
            loss= cross_entropy_loss(logits_reshaped, targets)

            val_loss_accumulated += loss.item()
            num_batches += 1

           
        val_loss_avg = val_loss_accumulated / num_batches

    print(f"Epoch {epoch+1}: Train Loss: {train_loss_avg:.4f}, Val Loss: {val_loss_avg:.4f}")
    # verify, if both decreases -> model is learning
    # if train loss decreases but val loss increases -> overfitting
    # if both losses are high and not decreasing -> underfitting
            
        


Epoch 1: Train Loss: 2.8152, Val Loss: 2.6886
Epoch 2: Train Loss: 2.6528, Val Loss: 2.5484
Epoch 3: Train Loss: 2.5574, Val Loss: 2.4632
Epoch 4: Train Loss: 2.5020, Val Loss: 2.4211
Epoch 5: Train Loss: 2.4673, Val Loss: 2.3905
Epoch 6: Train Loss: 2.4423, Val Loss: 2.3706
Epoch 7: Train Loss: 2.4233, Val Loss: 2.3526
Epoch 8: Train Loss: 2.4091, Val Loss: 2.3388
Epoch 9: Train Loss: 2.3972, Val Loss: 2.3332
Epoch 10: Train Loss: 2.3878, Val Loss: 2.3250
Epoch 11: Train Loss: 2.3795, Val Loss: 2.3166
Epoch 12: Train Loss: 2.3725, Val Loss: 2.3123
Epoch 13: Train Loss: 2.3659, Val Loss: 2.3094
Epoch 14: Train Loss: 2.3599, Val Loss: 2.3058
Epoch 15: Train Loss: 2.3540, Val Loss: 2.2993
Epoch 16: Train Loss: 2.3488, Val Loss: 2.2988
Epoch 17: Train Loss: 2.3446, Val Loss: 2.3004
Epoch 18: Train Loss: 2.3410, Val Loss: 2.2930
Epoch 19: Train Loss: 2.3375, Val Loss: 2.2957
Epoch 20: Train Loss: 2.3346, Val Loss: 2.2951
